# IUROPA network consistency checks — judgment-only universe

This notebook checks citation-network consistency **after restricting the decision universe to judgments only**.

The logic is:

1. load `decisions_curated.csv`;
2. keep only rows where `decision_type == "judgment"`;
3. attach decision metadata to both endpoints of `citations.csv`;
4. retain only citations where **both sender and receiver are judgments**;
5. test endpoint coverage and temporal consistency inside this judgment-to-judgment network.

So the resulting network is explicitly:

\[
	ext{judgment} ightarrow 	ext{judgment}
\]

No orders, AG opinions, AG views, opinions, decisions, or rulings are part of the retained citation network.

Nothing is silently deleted from the raw files: the notebook keeps diagnostic tables showing what was excluded.


In [1]:
# ============================================================
# 0. CONFIGURATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

CITATIONS_CSV = Path("citations.csv")
DECISIONS_CSV = Path("decisions_curated.csv")

for p in [CITATIONS_CSV, DECISIONS_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Put it in the same folder as this notebook."
        )

print("Inputs found:")
print(" ", CITATIONS_CSV.resolve())
print(" ", DECISIONS_CSV.resolve())


Inputs found:
  C:\Users\laze\Downloads\IUROPA\IUROPA\citations.csv
  C:\Users\laze\Downloads\IUROPA\IUROPA\decisions_curated.csv


In [2]:
# ============================================================
# 1. LOAD RAW TABLES
# ============================================================

citations_raw = pd.read_csv(CITATIONS_CSV)
decisions_raw = pd.read_csv(DECISIONS_CSV)

print("citations.csv shape:", citations_raw.shape)
print("decisions_curated.csv shape:", decisions_raw.shape)

required_citation_cols = {"citing_decision_id", "cited_decision_id"}
missing = required_citation_cols - set(citations_raw.columns)
if missing:
    raise KeyError(f"citations.csv missing required columns: {sorted(missing)}")

required_decision_cols = {"decision_id", "decision_type", "decision_date"}
missing = required_decision_cols - set(decisions_raw.columns)
if missing:
    raise KeyError(f"decisions_curated.csv missing required columns: {sorted(missing)}")


citations.csv shape: (283994, 2)
decisions_curated.csv shape: (56951, 12)


In [3]:
# ============================================================
# 2. RESTRICT NODE UNIVERSE TO JUDGMENTS
# ============================================================

decisions = decisions_raw.loc[
    decisions_raw["decision_type"].astype(str).str.strip().str.lower().eq("judgment")
].copy()

decisions["decision_date"] = pd.to_datetime(
    decisions["decision_date"],
    errors="coerce"
)

print("RAW DECISION UNIVERSE")
print("---------------------")
print("Rows:", len(decisions_raw))
print("Unique decision IDs:", decisions_raw["decision_id"].nunique())

print("\nJUDGMENT-ONLY NODE UNIVERSE")
print("---------------------------")
print("Rows:", len(decisions))
print("Unique judgment IDs:", decisions["decision_id"].nunique())
print("Duplicate judgment IDs:", decisions["decision_id"].duplicated().sum())
print("Missing judgment dates:", decisions["decision_date"].isna().sum())

if "court" in decisions.columns:
    print("\nJudgments by court:")
    display(
        decisions["court"]
        .value_counts(dropna=False)
        .rename_axis("court")
        .reset_index(name="n")
    )


RAW DECISION UNIVERSE
---------------------
Rows: 56951
Unique decision IDs: 56951

JUDGMENT-ONLY NODE UNIVERSE
---------------------------
Rows: 25778
Unique judgment IDs: 25778
Duplicate judgment IDs: 0
Missing judgment dates: 0

Judgments by court:


,court,n
0,Court of Justice,15329
1,General Court,9728
2,Civil Service Tribunal,721


In [4]:
# ============================================================
# 3. ATTACH DECISION TYPE TO BOTH CITATION ENDPOINTS
# ============================================================

endpoint_lookup_cols = ["decision_id", "decision_type", "decision_date"]
if "court" in decisions_raw.columns:
    endpoint_lookup_cols.append("court")

endpoint_lookup = decisions_raw[endpoint_lookup_cols].drop_duplicates("decision_id").copy()

sender_lookup = endpoint_lookup.rename(
    columns={
        "decision_id": "citing_decision_id",
        "decision_type": "citing_decision_type",
        "decision_date": "citing_decision_date",
        **({"court": "citing_court"} if "court" in endpoint_lookup.columns else {})
    }
)

receiver_lookup = endpoint_lookup.rename(
    columns={
        "decision_id": "cited_decision_id",
        "decision_type": "cited_decision_type",
        "decision_date": "cited_decision_date",
        **({"court": "cited_court"} if "court" in endpoint_lookup.columns else {})
    }
)

citations_labeled = (
    citations_raw
    .merge(
        sender_lookup,
        on="citing_decision_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        receiver_lookup,
        on="cited_decision_id",
        how="left",
        validate="many_to_one"
    )
)

print("Raw citation rows:", len(citations_labeled))

print("\nCiting endpoint types:")
display(
    citations_labeled["citing_decision_type"]
    .value_counts(dropna=False)
    .rename_axis("citing_decision_type")
    .reset_index(name="n")
)

print("\nCited endpoint types:")
display(
    citations_labeled["cited_decision_type"]
    .value_counts(dropna=False)
    .rename_axis("cited_decision_type")
    .reset_index(name="n")
)


Raw citation rows: 283994

Citing endpoint types:


,citing_decision_type,n
0,judgment,267095
1,NaN,16899



Cited endpoint types:


,cited_decision_type,n
0,judgment,283040
1,NaN,954


In [5]:
# ============================================================
# 4. RETAIN ONLY JUDGMENT -> JUDGMENT CITATIONS
# ============================================================

is_citing_judgment = (
    citations_labeled["citing_decision_type"]
    .astype(str).str.strip().str.lower().eq("judgment")
)

is_cited_judgment = (
    citations_labeled["cited_decision_type"]
    .astype(str).str.strip().str.lower().eq("judgment")
)

citations = citations_labeled.loc[
    is_citing_judgment & is_cited_judgment
].copy()

excluded_non_judgment_edges = citations_labeled.loc[
    ~(is_citing_judgment & is_cited_judgment)
].copy()

print("RAW CITATIONS")
print("-------------")
print("Rows:", len(citations_raw))
print("Unique citation pairs:",
      citations_raw[["citing_decision_id", "cited_decision_id"]]
      .drop_duplicates().shape[0])

print("\nJUDGMENT -> JUDGMENT CITATIONS")
print("------------------------------")
print("Rows:", len(citations))
print("Unique citation pairs:",
      citations[["citing_decision_id", "cited_decision_id"]]
      .drop_duplicates().shape[0])
print("Unique citing judgments:", citations["citing_decision_id"].nunique())
print("Unique cited judgments:", citations["cited_decision_id"].nunique())

print("\nExcluded because at least one endpoint is not a judgment:")
print(len(excluded_non_judgment_edges))


RAW CITATIONS
-------------
Rows: 283994
Unique citation pairs: 283994

JUDGMENT -> JUDGMENT CITATIONS
------------------------------
Rows: 266217
Unique citation pairs: 266217
Unique citing judgments: 22338
Unique cited judgments: 21700

Excluded because at least one endpoint is not a judgment:
17777


In [6]:
# ============================================================
# 5. ENDPOINT CONSISTENCY WITH THE JUDGMENT NODE UNIVERSE
# ============================================================

judgment_ids = set(decisions["decision_id"].dropna().astype(str))

citations["sender_in_judgment_nodes"] = (
    citations["citing_decision_id"].astype(str).isin(judgment_ids)
)
citations["receiver_in_judgment_nodes"] = (
    citations["cited_decision_id"].astype(str).isin(judgment_ids)
)
citations["both_endpoints_in_judgment_nodes"] = (
    citations["sender_in_judgment_nodes"]
    & citations["receiver_in_judgment_nodes"]
)

missing_sender_rows = citations.loc[
    ~citations["sender_in_judgment_nodes"]
].copy()

missing_receiver_rows = citations.loc[
    ~citations["receiver_in_judgment_nodes"]
].copy()

missing_any_rows = citations.loc[
    ~citations["both_endpoints_in_judgment_nodes"]
].copy()

print("ENDPOINT COVERAGE WITHIN JUDGMENT NODE UNIVERSE")
print("------------------------------------------------")
print("Sender found:",
      citations["sender_in_judgment_nodes"].sum(), "/", len(citations))
print("Receiver found:",
      citations["receiver_in_judgment_nodes"].sum(), "/", len(citations))
print("Both found:",
      citations["both_endpoints_in_judgment_nodes"].sum(), "/", len(citations))

print("\nRows with missing sender:", len(missing_sender_rows))
print("Rows with missing receiver:", len(missing_receiver_rows))
print("Rows with either endpoint missing:", len(missing_any_rows))


ENDPOINT COVERAGE WITHIN JUDGMENT NODE UNIVERSE
------------------------------------------------
Sender found: 266217 / 266217
Receiver found: 266217 / 266217
Both found: 266217 / 266217

Rows with missing sender: 0
Rows with missing receiver: 0
Rows with either endpoint missing: 0


In [7]:
# ============================================================
# 6. TEMPORAL CONSISTENCY IN THE JUDGMENT-ONLY NETWORK
# ============================================================

citations["citing_decision_date"] = pd.to_datetime(
    citations["citing_decision_date"],
    errors="coerce"
)
citations["cited_decision_date"] = pd.to_datetime(
    citations["cited_decision_date"],
    errors="coerce"
)

citations["citation_lag_days"] = (
    citations["citing_decision_date"]
    - citations["cited_decision_date"]
).dt.days

citations["is_self_citation"] = (
    citations["citing_decision_id"]
    == citations["cited_decision_id"]
)

citations["is_strict_past"] = (
    citations["cited_decision_date"]
    < citations["citing_decision_date"]
)

citations["is_same_day"] = (
    citations["cited_decision_date"]
    == citations["citing_decision_date"]
)

citations["is_future_citation"] = (
    citations["cited_decision_date"]
    > citations["citing_decision_date"]
)

citations["missing_sender_date"] = citations["citing_decision_date"].isna()
citations["missing_receiver_date"] = citations["cited_decision_date"].isna()

same_day = citations.loc[citations["is_same_day"]].copy()
future = citations.loc[citations["is_future_citation"]].copy()
self_citations = citations.loc[citations["is_self_citation"]].copy()

non_strict_past = citations.loc[
    citations["citation_lag_days"].notna()
    & (citations["citation_lag_days"] <= 0)
].copy()

print("TEMPORAL CONSISTENCY — JUDGMENT -> JUDGMENT")
print("-------------------------------------------")
print("Total judgment citations:", len(citations))
print("Strictly past:", int(citations["is_strict_past"].sum()))
print("Same day:", int(citations["is_same_day"].sum()))
print("Future-dated:", int(citations["is_future_citation"].sum()))
print("Self-citations:", int(citations["is_self_citation"].sum()))
print("Missing sender date:", int(citations["missing_sender_date"].sum()))
print("Missing receiver date:", int(citations["missing_receiver_date"].sum()))

print("\nCitation lag summary (days):")
display(citations["citation_lag_days"].describe())


TEMPORAL CONSISTENCY — JUDGMENT -> JUDGMENT
-------------------------------------------
Total judgment citations: 266217
Strictly past: 260252
Same day: 5040
Future-dated: 925
Self-citations: 2787
Missing sender date: 0
Missing receiver date: 0

Citation lag summary (days):


count    266217.000000
mean       2840.042725
std        2516.518374
min       -8364.000000
25%         960.000000
50%        2144.000000
75%        4026.000000
max       20944.000000
Name: citation_lag_days, dtype: float64

In [8]:
# ============================================================
# 7. INSPECT TEMPORAL PROBLEMS
# ============================================================

cols_to_show = [
    c for c in [
        "citing_decision_id",
        "cited_decision_id",
        "citing_decision_date",
        "cited_decision_date",
        "citation_lag_days",
        "citing_decision_type",
        "cited_decision_type",
        "citing_court",
        "cited_court",
    ]
    if c in citations.columns
]

print("Same-day judgment citations:", len(same_day))
display(same_day[cols_to_show].head(30))

print("\nFuture-dated judgment citations:", len(future))
display(future[cols_to_show].head(30))

print("\nSelf-citations:", len(self_citations))
display(self_citations[cols_to_show].head(30))


Same-day judgment citations: 5040


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citing_decision_type,cited_decision_type,citing_court,cited_court
49,DOC_JUDG_GC_2005_0456_20100428,DOC_JUDG_GC_2005_0456_20100428,2010-04-28,2010-04-28,0,judgment,judgment,General Court,General Court
124,DOC_JUDG_CJ_2002_0175_20050113,DOC_JUDG_CJ_2002_0174_20050113,2005-01-13,2005-01-13,0,judgment,judgment,Court of Justice,Court of Justice
240,DOC_JUDG_CJ_2022_0639_20240905,DOC_JUDG_CJ_2022_0639_20240905,2024-09-05,2024-09-05,0,judgment,judgment,Court of Justice,Court of Justice
287,DOC_JUDG_GC_1993_0432_19950307,DOC_JUDG_GC_1993_0432_19950307,1995-03-07,1995-03-07,0,judgment,judgment,General Court,General Court
365,DOC_JUDG_CJ_2018_0573_20191009,DOC_JUDG_CJ_2018_0573_20191009,2019-10-09,2019-10-09,0,judgment,judgment,Court of Justice,Court of Justice
384,DOC_JUDG_CJ_1982_0010_19830714,DOC_JUDG_CJ_1982_0010_19830714,1983-07-14,1983-07-14,0,judgment,judgment,Court of Justice,Court of Justice
399,DOC_JUDG_CJ_1987_0291_19881214,DOC_JUDG_CJ_1987_0291_19881214,1988-12-14,1988-12-14,0,judgment,judgment,Court of Justice,Court of Justice
400,DOC_JUDG_CJ_2024_0212_20250508,DOC_JUDG_CJ_2024_0212_20250508,2025-05-08,2025-05-08,0,judgment,judgment,Court of Justice,Court of Justice
429,DOC_JUDG_GC_2016_0836_20190516,DOC_JUDG_GC_2016_0836_20190516,2019-05-16,2019-05-16,0,judgment,judgment,General Court,General Court
459,DOC_JUDG_CST_2005_0022_20061213,DOC_JUDG_CST_2005_0022_20061213,2006-12-13,2006-12-13,0,judgment,judgment,Civil Service Tribunal,Civil Service Tribunal



Future-dated judgment citations: 925


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citing_decision_type,cited_decision_type,citing_court,cited_court
1300,DOC_JUDG_CST_2007_0114_20090929,DOC_JUDG_GC_2009_0091_P_20100901,2009-09-29,2010-09-01,-337,judgment,judgment,Civil Service Tribunal,General Court
1376,DOC_JUDG_CST_2008_0052_20090604,DOC_JUDG_GC_2006_0404_P_20090908,2009-06-04,2009-09-08,-96,judgment,judgment,Civil Service Tribunal,General Court
1612,DOC_JUDG_CST_2014_0132_20151006,DOC_JUDG_GC_2015_0309_P_20161027,2015-10-06,2016-10-27,-387,judgment,judgment,Civil Service Tribunal,General Court
1884,DOC_JUDG_CST_2010_0062_20110915,DOC_JUDG_GC_2009_0325_P_20110921,2011-09-15,2011-09-21,-6,judgment,judgment,Civil Service Tribunal,General Court
1920,DOC_JUDG_GC_2016_0639_P_20180123,DOC_JUDG_GC_2016_0750_20181214,2018-01-23,2018-12-14,-325,judgment,judgment,General Court,General Court
2444,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_GC_1993_0485_20001108,1996-09-24,2000-11-08,-1506,judgment,judgment,General Court,General Court
2447,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_CJ_1996_0386_P_19980505,1996-09-24,1998-05-05,-588,judgment,judgment,General Court,Court of Justice
2448,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_CJ_1996_0391_P_19980505,1996-09-24,1998-05-05,-588,judgment,judgment,General Court,Court of Justice
2449,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_CJ_1996_0403_P_19980505,1996-09-24,1998-05-05,-588,judgment,judgment,General Court,Court of Justice
2454,DOC_JUDG_GC_1993_0485_19960924,DOC_JUDG_GC_1997_0266_19990708,1996-09-24,1999-07-08,-1017,judgment,judgment,General Court,General Court



Self-citations: 2787


,citing_decision_id,cited_decision_id,citing_decision_date,cited_decision_date,citation_lag_days,citing_decision_type,cited_decision_type,citing_court,cited_court
49,DOC_JUDG_GC_2005_0456_20100428,DOC_JUDG_GC_2005_0456_20100428,2010-04-28,2010-04-28,0,judgment,judgment,General Court,General Court
240,DOC_JUDG_CJ_2022_0639_20240905,DOC_JUDG_CJ_2022_0639_20240905,2024-09-05,2024-09-05,0,judgment,judgment,Court of Justice,Court of Justice
287,DOC_JUDG_GC_1993_0432_19950307,DOC_JUDG_GC_1993_0432_19950307,1995-03-07,1995-03-07,0,judgment,judgment,General Court,General Court
365,DOC_JUDG_CJ_2018_0573_20191009,DOC_JUDG_CJ_2018_0573_20191009,2019-10-09,2019-10-09,0,judgment,judgment,Court of Justice,Court of Justice
384,DOC_JUDG_CJ_1982_0010_19830714,DOC_JUDG_CJ_1982_0010_19830714,1983-07-14,1983-07-14,0,judgment,judgment,Court of Justice,Court of Justice
399,DOC_JUDG_CJ_1987_0291_19881214,DOC_JUDG_CJ_1987_0291_19881214,1988-12-14,1988-12-14,0,judgment,judgment,Court of Justice,Court of Justice
400,DOC_JUDG_CJ_2024_0212_20250508,DOC_JUDG_CJ_2024_0212_20250508,2025-05-08,2025-05-08,0,judgment,judgment,Court of Justice,Court of Justice
429,DOC_JUDG_GC_2016_0836_20190516,DOC_JUDG_GC_2016_0836_20190516,2019-05-16,2019-05-16,0,judgment,judgment,General Court,General Court
459,DOC_JUDG_CST_2005_0022_20061213,DOC_JUDG_CST_2005_0022_20061213,2006-12-13,2006-12-13,0,judgment,judgment,Civil Service Tribunal,Civil Service Tribunal
523,DOC_JUDG_GC_2013_0429_20180517,DOC_JUDG_GC_2013_0429_20180517,2018-05-17,2018-05-17,0,judgment,judgment,General Court,General Court


In [9]:
# ============================================================
# 8. COURT PATTERNS IN PROBLEMATIC JUDGMENT EDGES
# ============================================================

if {"citing_court", "cited_court"}.issubset(citations.columns):
    print("ALL JUDGMENT CITATIONS: sender x receiver court")
    display(
        pd.crosstab(
            citations["citing_court"],
            citations["cited_court"],
            margins=True,
            dropna=False,
        )
    )

    if len(non_strict_past):
        print("\nNON-STRICT-PAST JUDGMENT CITATIONS: sender x receiver court")
        display(
            pd.crosstab(
                non_strict_past["citing_court"],
                non_strict_past["cited_court"],
                margins=True,
                dropna=False,
            )
        )


ALL JUDGMENT CITATIONS: sender x receiver court


cited_court,Civil Service Tribunal,Court of Justice,General Court,All
citing_court,,,,
Civil Service Tribunal,3188,1650,5524,10362
Court of Justice,31,88343,3303,91677
General Court,1628,63812,98738,164178
All,4847,153805,107565,266217



NON-STRICT-PAST JUDGMENT CITATIONS: sender x receiver court


cited_court,Civil Service Tribunal,Court of Justice,General Court,All
citing_court,,,,
Civil Service Tribunal,281,14,192,487
Court of Justice,1,2315,97,2413
General Court,2,160,2903,3065
All,284,2489,3192,5965


In [10]:
# ============================================================
# 9. DIAGNOSTIC VALID-REM-EVENT FLAG
# ============================================================
#
# Still diagnostic only: nothing is dropped automatically.

citations["valid_rem_event_candidate"] = (
    citations["both_endpoints_in_judgment_nodes"]
    & ~citations["is_self_citation"]
    & citations["is_strict_past"]
)

summary = pd.DataFrame({
    "metric": [
        "raw_decisions",
        "judgment_nodes",
        "raw_citations",
        "judgment_to_judgment_citations",
        "excluded_non_judgment_edges",
        "judgment_edges_both_endpoints_found",
        "judgment_self_citations",
        "judgment_same_day_citations",
        "judgment_future_dated_citations",
        "judgment_strict_past_citations",
        "valid_rem_event_candidate",
    ],
    "n": [
        len(decisions_raw),
        len(decisions),
        len(citations_raw),
        len(citations),
        len(excluded_non_judgment_edges),
        int(citations["both_endpoints_in_judgment_nodes"].sum()),
        int(citations["is_self_citation"].sum()),
        int(citations["is_same_day"].sum()),
        int(citations["is_future_citation"].sum()),
        int(citations["is_strict_past"].sum()),
        int(citations["valid_rem_event_candidate"].sum()),
    ],
})

display(summary)


,metric,n
0,raw_decisions,56951
1,judgment_nodes,25778
2,raw_citations,283994
3,judgment_to_judgment_citations,266217
4,excluded_non_judgment_edges,17777
5,judgment_edges_both_endpoints_found,266217
6,judgment_self_citations,2787
7,judgment_same_day_citations,5040
8,judgment_future_dated_citations,925
9,judgment_strict_past_citations,260252


In [11]:
# ============================================================
# 10. OPTIONAL DIAGNOSTIC EXPORTS
# ============================================================

summary.to_csv(
    "network_consistency_v2_summary.csv",
    index=False
)

excluded_non_judgment_edges.to_csv(
    "network_consistency_v2_excluded_non_judgment_edges.csv",
    index=False
)

missing_any_rows.to_csv(
    "network_consistency_v2_missing_judgment_endpoints.csv",
    index=False
)

non_strict_past.to_csv(
    "network_consistency_v2_non_strict_past.csv",
    index=False
)

print("Saved:")
print(" network_consistency_v2_summary.csv")
print(" network_consistency_v2_excluded_non_judgment_edges.csv")
print(" network_consistency_v2_missing_judgment_endpoints.csv")
print(" network_consistency_v2_non_strict_past.csv")


Saved:
 network_consistency_v2_summary.csv
 network_consistency_v2_excluded_non_judgment_edges.csv
 network_consistency_v2_missing_judgment_endpoints.csv
 network_consistency_v2_non_strict_past.csv
